### Load converters

In [78]:
%reload_ext autoreload
%autoreload 0

import importlib
import os
from pathlib import Path

# TOM assemblies live at the package root, but this notebook's cwd is tests/.
os.environ["OSSIE_MICROSOFT_TOM_ASSEMBLIES"] = str(
    Path.cwd().parent / ".tom" / "assemblies"
)

_sql_to_dax = importlib.reload(
    importlib.import_module("ossie_microsoft._sql_to_dax")
)
_tom = importlib.reload(importlib.import_module("ossie_microsoft.tom"))
_converter = importlib.reload(
    importlib.import_module("ossie_microsoft.ossie_to_semantic_model")
)
convert_ossie_to_semantic_model = _converter.convert_ossie_to_semantic_model
convert_semantic_model_to_ossie = importlib.reload(
    importlib.import_module("ossie_microsoft.semantic_model_to_ossie")
).convert_semantic_model_to_ossie

print("Loaded converters!")


Loaded converters!


### Convert Ossie to Semantic Model

Set `use_sql_measures = True` and `output_format = "TMSL"` below to restore retained SQL measure expressions instead of generated DAX. This requires a preview runtime supporting SQL measures at compatibility level `1000000`; ordinary Power BI deployments are not supported. The public TOM serializer does not yet support SQL measure metadata, so this mode cannot emit TMDL.

Only measures annotated by the converter with source dialect `ANSI_SQL` or `OSSIE_SQL_2026` are promoted to destination `dialect: sql`. The full retained expression is preserved, not translated or checked for runtime SQL support. SQL mode restores retained SQL annotations even if the converter selected DAX, so historical SQL annotations can replace subsequently edited DAX expressions. Measures without SQL annotations, including unannotated DAX, remain unchanged. Other source dialects retain the converter output with a warning. Missing expressions retain the converter's warning-and-skip behavior.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/apache/ossie/main/examples/tpcds_semantic_model.yaml"

response = requests.get(url)
osi = response.text
#print(osi)

In [ ]:
import warnings

workspace_id = "00000000-0000-0000-0000-000000000000" # ID of workspace
item_id = "00000000-0000-0000-0000-000000000000" # ID of Lakehouse
output_format = "TMDL"  # Can be "TMSL" or "TMDL"
use_sql_measures = False

if use_sql_measures and output_format != "TMSL":
    raise ValueError('SQL measures require output_format = "TMSL"; public TOM cannot emit them.')

x = convert_ossie_to_semantic_model(
    ossie_yaml_str=osi,
    source={
        "workspaceId": workspace_id,
        "itemId": item_id,
    },
    output_format=output_format
)

if use_sql_measures:
    for table in x["model"]["tables"]:
        for measure in table.get("measures", []):
            annotations = {a["name"]: a["value"] for a in measure.get("annotations", [])}
            source_dialect = annotations.get("OssieExpressionDialect")
            if source_dialect is None:
                continue
            if source_dialect not in ("ANSI_SQL", "OSSIE_SQL_2026"):
                warnings.warn(
                    f"Measure '{table['name']}.{measure['name']}': source dialect "
                    f"'{source_dialect}' is not promoted to SQL; retaining converter output.",
                    stacklevel=2,
                )
                continue
            measure["expression"] = annotations["OssieExpression"]
            measure["dialect"] = "sql"
            x["compatibilityLevel"] = max(x["compatibilityLevel"], 1000000)

In [ ]:
import json

if output_format == "TMSL":
    print(json.dumps(x, indent=2))
else:
    print(x)

### Convert Semantic Model to Ossie

In [ ]:
import json

with open(Path.cwd() / "fixtures" / "sales_model.bim", encoding="utf-8-sig") as fh:
    bim = json.load(fh)

with open(Path.cwd() / "fixtures" / "sales_model.tmdl", encoding="utf-8-sig") as fh:
    tmdl = fh.read()

In [ ]:
ossie = convert_semantic_model_to_ossie(
    semantic_model=tmdl # or bim
)

In [ ]:
print(ossie)